# 03 — Platform influence in the ADCP–satellite example

**Question:** How do the fitted ESP parameters change when ADCP and satellite influence is specified explicitly in the inner and outer stages?

Compare ADCP only, satellite only, the original equal-per-observation combined fit, and a combined fit with separate platform weights. Defaults give ADCP all inner-stage influence and both platforms equal *mean-error* contributions outside. Sweep each weight separately. This is one fit to the original observation window, not a time series.

All observations within the fitting domains remain available; there is no thinning. Original inner fits import `ESP_zonodo/functions.py`. Weighted variants and explicit outer-fit diagnostics live locally in `platform_weighting.py`; **ESP_zonodo is never edited**. Restart the kernel and run in order.

In [ ]:
from pathlib import Path

SAVE_OUTPUTS = False  # CSV, PNG and provenance files; Jupyter can still save cell outputs
ESP_ROOT = Path('/home/z5297792/ESP_zonodo')
DATA_DIR = ESP_ROOT / 'data'
ADCP_FILE = DATA_DIR / 'sadcp_data.mat'
SATELLITE_FILE = DATA_DIR / 'satellite_data.nc'

ADCP_START = '2023-10-17'  # inclusive; retains the original example's southern transects
ADCP_END = '2023-10-22'    # exclusive
ADCP_DEPTH_M = 37.0
LATITUDE_LIMIT = -37.0
LON0, LAT0 = 154.3, -37.9
CORE_RADIUS_M = 35_000.0  # original code default; manuscript currently describes 30 km
OUTER_RADIUS_M = 100_000.0
RC_LIMIT_M = 1_000_000.0  # numerical safeguard, separate from observation-selection radius

SAT_WEIGHT_INNER = 0.0  # 0 excludes satellite here; 1 gives equal platform mean-error terms
SAT_WEIGHT_OUTER = 1.0
WEIGHT_SWEEP = [0.0, 0.1, 0.3, 1.0, 3.0]
OUTPUT_ROOT = Path('results/03_platform_weighting')

For either stage, the weighted objective is

\[
J=\frac{1}{N_A}\sum_{i\in A}e_i^2+
\lambda\frac{1}{N_S}\sum_{i\in S}e_i^2.
\]

The inner residual is the two-component **linear-core** velocity error. The outer residual is the original **normalised tangential Gaussian-profile** error. Each stage has its own counts and satellite weight; the ADCP multiplier is 1. Outer counts are measured after the original rotation-sign filter. Zero-weight observations do not enter the fit or its initialisation.

- `0`: ADCP only in that stage; `0.1`: satellite contributes one tenth of the ADCP *mean-error term*; `1`: equal platform terms.
- Equal per-observation weighting corresponds to `lambda = N_satellite / N_ADCP`, separately for each stage. It does **not** generally correspond to 1.
- These are influence choices, not measurement-error estimates. Normalisation removes total point-count scaling between platforms, but does not remove within-platform clustering or correlation, or establish an effective sample size.

The three original fits use their own original two-pass selection. For the selected weighted fit and both sweeps, freeze the original combined fit's inner and outer observation sets. This isolates weighting from moving selection boundaries. The original outer rotation-sign filter still applies and can change with fitted geometry; retained counts are reported. Thus the ADCP-led weighted fit need not exactly equal the separately selected ADCP-only fit.

In [ ]:
import hashlib
import json
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import validation_helpers as vh
import platform_weighting as pw

backend, backend_info = vh.load_backend(ESP_ROOT)
observations, data_info = pw.load_observations(
    ADCP_FILE, SATELLITE_FILE, start=ADCP_START, end=ADCP_END,
    depth=ADCP_DEPTH_M, latitude_limit=LATITUDE_LIMIT, lon0=LON0, lat0=LAT0)
settings = dict(core_radius=CORE_RADIUS_M, outer_radius=OUTER_RADIUS_M, rc_limit=RC_LIMIT_M)
print('Backend:', backend_info)
print('ADCP mean/reference time (UTC):', data_info['reference_time'])
print('Actual ADCP depth (m):', data_info['actual_adcp_depth_m'])
print('Native satellite fields used (UTC):', data_info['satellite_native_used'])
print('Daily satellite interpolation bracket:', data_info['satellite_daily_bracket'])
display(pw.support_table(observations, observations.id, 'available before spatial selection'))

The original ADCP window spans approximately 4.5 days. Preserve the original satellite processing: native fields → daily 06:00 UTC → the ADCP mean time. Satellite rows below have this common analysis timestamp, **not independent acquisition times**; the contributing native timestamps are printed above. Surface geostrophic satellite velocity and total ADCP velocity at approximately 37 m remain physically different. Weighting does not correct ageostrophic flow, vertical shear, bias, or eddy evolution over the window.

The 35 km inner radius follows the released code. Reconcile the manuscript's 30 km description when revising; this notebook does not silently change that setting.

In [ ]:
originals = [
    pw.fit(backend, observations, 'ADCP only', source='ADCP', **settings),
    pw.fit(backend, observations, 'Satellite only', source='Satellite', **settings),
    pw.fit(backend, observations, 'Original combined', **settings),
]
reference = originals[-1]
display(pd.DataFrame([f['summary'] for f in originals])[
    ['label', 'status', 'inner_valid', 'valid']])
if not reference['summary']['valid']:
    raise RuntimeError('The original combined fit failed; inspect its status before sweeping weights.')

# Check the local formulations against the actual imported functions on these data.
checks = []
for f in originals:
    if not f['summary']['valid']:
        continue
    core = observations[observations.id.isin(f['inner_ids'])]
    counts = core.platform.value_counts()
    equivalent_weight = counts.get('Satellite', 0) / counts.get('ADCP', 1)
    local = pw.weighted_inner(core, equivalent_weight)
    inner = f['inner']
    np.testing.assert_allclose([local['xc'], local['yc']], [inner['xc'], inner['yc']], atol=0.1, rtol=0)
    np.testing.assert_allclose(local['Q'], inner['Q'], atol=1e-5, rtol=1e-5)
    np.testing.assert_allclose(local['Omega'], inner['Omega'], atol=1e-10, rtol=1e-5)
    outer = observations[observations.id.isin(f['outer_ids'])]
    rc, psi, omega = backend.out_core_param_fit(
        *[outer[k].to_numpy() for k in ('x','y','u','v')],
        inner['xc'], inner['yc'], inner['Q'], Omega0=inner['Omega'], Rc_max=RC_LIMIT_M)
    np.testing.assert_allclose([f['summary']['Rc_km']*1000, f['summary']['Omega']],
                               [rc, omega], rtol=1e-5, atol=1e-10)
    checks.append(dict(label=f['summary']['label'], inner_and_outer_backend_check='passed'))
display(pd.DataFrame(checks))

fixed_masks = {key: reference[key] for key in ('inner_ids', 'outer_ids')}
evaluation_ids = sorted(set(reference['inner_ids']) | set(reference['outer_ids']))
evaluation = observations[observations.id.isin(evaluation_ids)].copy()

## Compare the four fits

Residuals use the **same fixed observations for every fit**, reported separately by platform. They overlap fitting data and are descriptive residuals, not held-out validation or errors against truth. An ADCP-only fit's satellite residual describes disagreement between platforms, not independent ground truth. No combined score selects a “best” weight.

The inner weight changes centre and ellipse geometry. The outer fit refines both Gaussian radius and rotation amplitude, so the final vorticity can depend on satellite observations even when the inner satellite weight is zero.

In [ ]:
selected = pw.fit(backend, observations, 'Selected weighted', mode='weighted',
    inner_weight=SAT_WEIGHT_INNER, outer_weight=SAT_WEIGHT_OUTER,
    fixed_masks=fixed_masks, **settings)
comparisons = originals + [selected]
parameter_columns = ['label','status','xc_km','yc_km','alpha','angle_deg','Rc_km',
                     'inner_vorticity','vorticity','peak_bracketed']
comparison_table = pd.DataFrame([f['summary'] for f in comparisons])
display(comparison_table.reindex(columns=parameter_columns))
display(comparison_table.filter(regex=r'^label$|^n_'))
comparison_residuals = pw.residuals(comparisons, evaluation)
display(comparison_residuals)
fig_comparison = pw.plot_comparison(comparisons, evaluation)
plt.show()

`alpha` is the major/minor axis ratio; orientation is axial (modulo 180°). `Rc` is the Gaussian scale, whose speed maximum lies at `Rc / sqrt(2)`. Vorticity units are s⁻¹. `peak_bracketed` only checks whether retained effective radii lie on both sides of that maximum; it does not demonstrate full spatial coverage or parameter certainty.

“Valid” means passing numerical and geometry checks, not scientific accuracy. A failed outer fit retains any available centre/geometry, but is excluded from full-model residuals. A local outer wrapper exposes failure instead of returning the initial guess silently.

In [ ]:
weights = sorted(set([float(w) for w in WEIGHT_SWEEP]))
if not weights or not np.isfinite(weights).all() or min(weights) < 0:
    raise ValueError('WEIGHT_SWEEP must contain finite nonnegative values.')
inner_sweep = [pw.fit(backend, observations, f'Inner lambda={w:g}', mode='weighted',
    inner_weight=w, outer_weight=SAT_WEIGHT_OUTER, fixed_masks=fixed_masks, **settings) for w in weights]
outer_sweep = [pw.fit(backend, observations, f'Outer lambda={w:g}', mode='weighted',
    inner_weight=SAT_WEIGHT_INNER, outer_weight=w, fixed_masks=fixed_masks, **settings) for w in weights]
all_fits = comparisons + inner_sweep + outer_sweep
all_parameters = pd.DataFrame([f['summary'] for f in all_fits])
sweep_columns = ['label','satellite_weight_inner','satellite_weight_outer'] + parameter_columns[1:]
display(all_parameters.iloc[len(comparisons):].reindex(columns=sweep_columns))
fig_inner = pw.plot_sweep(inner_sweep, 'inner', reference['summary'], evaluation)
plt.show()
fig_outer = pw.plot_sweep(outer_sweep, 'outer', reference['summary'], evaluation)
plt.show()

## Sample support and interpretation

The table below audits selected/active counts, time span, depth and objective weight per platform in **every fit and stage**. For the inner stage, selected includes zero-weight satellite rows; active excludes them. For the outer stage, selected already reflects the sign filter and removal of zero-weight rows. The earlier comparison table also reports outer counts before those removals. Satellite time spans describe the interpolated snapshot; native time support is recorded above and in provenance. Observation uncertainties are not estimated by this experiment.

Read these results as a sensitivity test:

1. Does the ADCP-led inner fit retain a similar centre and shape as satellite inner influence increases?
2. Does adding satellite outer influence change `Rc`, vorticity, and the ADCP/satellite residual trade-off appreciably?
3. Are differences supported by each platform's sampling coverage, or accompanied by failures/poor radial support?

Stable parameters support robustness **over the tested weights and window**. Large shifts show application dependence; neither result establishes which parameters are true. These experiments address raw platform-count influence and disclose the trade-off. They do not fully resolve the reviewer's correlation, uncertainty or physical-harmonisation concerns, and should not be described as doing so.

In [ ]:
audit = pw.fit_support(all_fits, observations)
all_residuals = pw.residuals(all_fits, evaluation)
with pd.option_context('display.max_rows', len(audit)):
    display(audit)
failures_or_warnings = all_parameters[(~all_parameters.valid) | (all_parameters.warnings != '')]
if len(failures_or_warnings):
    display(failures_or_warnings.reindex(columns=['label','status','message','warnings']))
else:
    print('All attempted fits passed numerical checks; no recorded fitting warnings.')

In [ ]:
if SAVE_OUTPUTS:
    run_dir = OUTPUT_ROOT / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    run_dir.mkdir(parents=True, exist_ok=False)
    all_parameters.to_csv(run_dir / 'parameters.csv', index=False)
    all_residuals.to_csv(run_dir / 'common_set_residuals.csv', index=False)
    audit.to_csv(run_dir / 'sample_support.csv', index=False)
    observations.to_csv(run_dir / 'observations.csv', index=False)
    membership = [dict(label=f['summary']['label'], stage=stage, observation_id=i)
                  for f in all_fits for stage in ('inner_ids','outer_ids','outer_kept_ids') for i in f[stage]]
    pd.DataFrame(membership).to_csv(run_dir / 'fit_membership.csv', index=False)
    for name, fig in [('comparison',fig_comparison),('inner_weights',fig_inner),('outer_weights',fig_outer)]:
        fig.savefig(run_dir / f'{name}.png', dpi=180, bbox_inches='tight')
    controls = dict(ADCP_START=ADCP_START, ADCP_END=ADCP_END, ADCP_DEPTH_M=ADCP_DEPTH_M,
        LATITUDE_LIMIT=LATITUDE_LIMIT, LON0=LON0, LAT0=LAT0, **settings,
        SAT_WEIGHT_INNER=SAT_WEIGHT_INNER, SAT_WEIGHT_OUTER=SAT_WEIGHT_OUTER, WEIGHT_SWEEP=weights)
    local_paths = [Path(vh.__file__), Path(pw.__file__), Path('03_platform_weighting.ipynb')]
    provenance = dict(backend=backend_info, data=data_info, controls=controls,
        checks=checks, fixed_masks=fixed_masks, evaluation_ids=evaluation_ids,
        local_sha256={str(p.resolve()):hashlib.sha256(p.read_bytes()).hexdigest() for p in local_paths},
        interpretation='Application-specific influence sensitivity, not uncertainty estimation or independent validation.')
    (run_dir / 'provenance.json').write_text(json.dumps(provenance, indent=2, default=str))
    print('Saved:', run_dir.resolve())
else:
    print('SAVE_OUTPUTS=False: no result files written. Tables/figures remain visible in this notebook.')